# Load the data

In [1]:
import numpy as np
from pathlib import Path


def load_dataset(path="/home/gmarihuan/processed/chbmit_windows_all.npz"):
    """Return the whole dataset as X (float32), y, patient."""
    d = np.load(path, allow_pickle=True)
    X = d["X"].astype(np.float32)
    y = d["y"]
    patient = d["patient_id"]
    return X, y, patient


X, y, patient = load_dataset()

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpatialTemporalBranch(nn.Module):
    def __init__(self, in_channels):
        super(SpatialTemporalBranch, self).__init__()
        # The paper specifies a kernel size of (1, 4), stride of 2, and no padding[cite: 1].
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=(1, 4), stride=(1, 2), padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.act1 = nn.LeakyReLU()
        
        self.conv2 = nn.Conv2d(32, 32, kernel_size=(1, 4), stride=(1, 2), padding=0, bias=False)
        self.bn2 = nn.BatchNorm2d(32)
        self.act2 = nn.LeakyReLU()
        
        self.conv3 = nn.Conv2d(32, 32, kernel_size=(1, 4), stride=(1, 2), padding=0, bias=False)
        self.bn3 = nn.BatchNorm2d(32)
        self.act3 = nn.LeakyReLU()
        
        # Global average pooling condenses the output into 32 features[cite: 1].
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def _pad_temporal(self, x):
        """
        Dynamically pads the temporal dimension if it becomes smaller than the kernel size (4).
        Expected input shape: (Batch, Channels, Height, Time)
        """
        time_dim = x.size(3)
        if time_dim < 4:
            pad_amount = 4 - time_dim
            # F.pad format for 4D tensor: (pad_left, pad_right, pad_top, pad_bottom)
            x = F.pad(x, (0, pad_amount, 0, 0))
        return x

    def forward(self, x):
        x = self._pad_temporal(x)
        x = self.act1(self.bn1(self.conv1(x)))
        
        x = self._pad_temporal(x)
        x = self.act2(self.bn2(self.conv2(x)))
        
        x = self._pad_temporal(x)
        x = self.act3(self.bn3(self.conv3(x)))
        
        x = self.pool(x)
        return x

class EEGWaveNet(nn.Module):
    def __init__(self, in_channels=21, timepoints=768):
        super(EEGWaveNet, self).__init__()
        self.timepoints = timepoints
        
        # 1. Multiscale Convolution Module[cite: 1]
        # Consists of 6 consecutive depth-wise convolution layers[cite: 1].
        self.multiscale_layers = nn.ModuleList([
            nn.Conv2d(
                in_channels=in_channels, 
                out_channels=in_channels, 
                kernel_size=(1, 2), 
                stride=(1, 2), 
                groups=in_channels, 
                bias=False
            ) for _ in range(6)
        ])
        
        # 2. Spatial-Temporal Feature Extraction Module[cite: 1]
        # 5 parallel branches to process the outputs from the 2nd to 6th layers of the Multiscale Convolution[cite: 1].
        self.st_branches = nn.ModuleList([
            SpatialTemporalBranch(in_channels) for _ in range(5)
        ])
        
        # 3. Classifier Module[cite: 1]
        # Fully connected layers with sizes 160, 64, and 32, concluding with a Log Softmax classifier for 2 classes[cite: 1].
        self.classifier = nn.Sequential(
            nn.Linear(160, 64),
            nn.LeakyReLU(),
            
            nn.Linear(64, 32),
            nn.Sigmoid(),
            
            nn.Linear(32, 2),
            nn.LogSoftmax(dim=1)
        )

    def forward(self, x):
        # Expected input shape: (Batch_Size, Channels, 1, Timepoints)
        ms_outputs = []
        
        # Pass input through the Multiscale Convolution Module
        for i, layer in enumerate(self.multiscale_layers):
            x = layer(x)
            # Collect the outputs from the 2nd to 6th layers (indices 1 through 5)[cite: 1]
            if i >= 1:
                ms_outputs.append(x)
                
        st_outputs = []
        
        # Extract features from each scale
        for i, branch in enumerate(self.st_branches):
            branch_out = branch(ms_outputs[i])
            # Flatten the pooled output from (Batch_Size, 32, 1, 1) to (Batch_Size, 32)
            branch_out = torch.flatten(branch_out, 1)
            st_outputs.append(branch_out)
            
        # Concatenate the 32 features from all 5 scales to form an array of 160 features[cite: 1]
        features = torch.cat(st_outputs, dim=1)
        
        # Pass through the classifier
        out = self.classifier(features)
        
        return out

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, 
                epochs: int = 50, learning_rate: float = 0.001, patience: int = 5):
    
    # 1. Device Configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # 2. Loss Function and Optimizer
    # Defaulting to CrossEntropyLoss for classification; adjust as needed for regression, etc.
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Early stopping variables
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    # 3. Main Training Loop
    for epoch in range(epochs):
        # -- Training Phase --
        model.train() 
        running_loss = 0.0
        
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()            # Clear old gradients
            outputs = model(inputs)          # Forward pass
            loss = criterion(outputs, targets) # Compute loss
            
            loss.backward()                  # Backpropagation
            optimizer.step()                 # Update weights
            
            running_loss += loss.item() * inputs.size(0)
            
        train_loss = running_loss / len(train_loader.dataset)
        
        # -- Validation Phase --
        model.eval() 
        val_loss = 0.0
        
        with torch.no_grad(): # Disable gradient calculation for efficiency
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item() * inputs.size(0)
                
        val_loss = val_loss / len(val_loader.dataset)
        
        print(f"Epoch {epoch+1:03d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
        # 4. Early Stopping & Checkpointing
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break

    # 5. Restore Best Model
    print("Training complete. Loading the best model weights.")
    model.load_state_dict(torch.load('best_model.pth', weights_only=True))
    return model

In [4]:
import numpy as np
from sklearn.model_selection import train_test_split

def get_split(X, y, patient, test_patients, val_size=0.2, random_state=42):
    """
    Splits the dataset into train, validation, and test sets.
    The test set contains only the data from the specified test_patients.
    """
    # Ensure inputs are numpy arrays for boolean masking
    X = np.array(X)
    y = np.array(y)
    patient = np.array(patient)
    
    # 1. Create a mask for the test patients and isolate the test set
    test_mask = np.isin(patient, test_patients)
    X_test, y_test = X[test_mask], y[test_mask]
    
    # 2. Isolate the remaining data to be split into train and val
    train_val_mask = ~test_mask
    X_train_val = X[train_val_mask]
    y_train_val = y[train_val_mask]
    
    # 3. Split the remaining data into train and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, 
        y_train_val, 
        test_size=val_size, 
        random_state=random_state,
        stratify=y_train_val  # Maintains class distribution in train and val
    )
    
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

# --- Example Usage ---
# train_data, val_data, test_data = get_split(X, y, patient, test_patients=[1])
# X_train, y_train = train_data

In [5]:
def remove_patients(X, y, patient, exclude_patients):
    """
    Removes all records belonging to the specified patients from the dataset.
    """
    X = np.array(X)
    y = np.array(y)
    patient = np.array(patient)
    
    # Create a mask for rows where the patient is NOT in the exclude list
    keep_mask = ~np.isin(patient, exclude_patients)
    
    # Filter the arrays
    X_filtered = X[keep_mask]
    y_filtered = y[keep_mask]
    patient_filtered = patient[keep_mask]
    
    return X_filtered, y_filtered, patient_filtered

# --- Example Usage ---
# X_clean, y_clean, patient_clean = remove_patients(X, y, patient, exclude_patients=[0, 1, 2])

In [6]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Assuming your data is already split:
# (X_train, y_train), (X_val, y_val), (X_test, y_test) = get_split(X, y, patient, test_patients=[1])

def create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size=512):
    """
    Converts split data arrays into PyTorch DataLoaders.
    """
    # 1. Convert NumPy arrays to PyTorch Tensors
    # Features are typically float32, and classification labels are long (integers)
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.long)
    
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(2)
    X_val_t = torch.tensor(X_val, dtype=torch.float32).unsqueeze(2)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(2)
    
    # 2. Wrap Tensors in a TensorDataset
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    test_dataset = TensorDataset(X_test_t, y_test_t)
    
    # 3. Create DataLoaders
    # We shuffle the training data to prevent the model from learning sequence patterns,
    # but we don't need to shuffle validation or test sets.
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader

# --- Example Usage ---
# train_loader, val_loader, test_loader = create_dataloaders(
#     X_train, y_train, 
#     X_val, y_val, 
#     X_test, y_test, 
#     batch_size=512
# )

In [7]:
X[0].shape

(21, 768)

In [8]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = get_split(X, y, patient, test_patients=[0])
train_loader, val_loader, test_loader = create_dataloaders(
     X_train, y_train, 
     X_val, y_val, 
     X_test, y_test,      
     batch_size=512
)

model = EEGWaveNet(21, timepoints=768)
trained_model = train_model(model=model, train_loader=train_loader, val_loader=val_loader)

Epoch 001/50 | Train Loss: 0.6632 | Val Loss: 0.6441
Epoch 002/50 | Train Loss: 0.5972 | Val Loss: 0.5932
Epoch 003/50 | Train Loss: 0.5494 | Val Loss: 0.5676
Epoch 004/50 | Train Loss: 0.5101 | Val Loss: 0.5871
Epoch 005/50 | Train Loss: 0.4426 | Val Loss: 0.6139
Epoch 006/50 | Train Loss: 0.3500 | Val Loss: 0.6778
Epoch 007/50 | Train Loss: 0.2750 | Val Loss: 0.5390
Epoch 008/50 | Train Loss: 0.2265 | Val Loss: 0.5219
Epoch 009/50 | Train Loss: 0.1775 | Val Loss: 0.4012
Epoch 010/50 | Train Loss: 0.1395 | Val Loss: 0.3549
Epoch 011/50 | Train Loss: 0.1162 | Val Loss: 0.3653
Epoch 012/50 | Train Loss: 0.1014 | Val Loss: 0.3813
Epoch 013/50 | Train Loss: 0.0969 | Val Loss: 0.3961
Epoch 014/50 | Train Loss: 0.0813 | Val Loss: 0.4024
Epoch 015/50 | Train Loss: 0.0669 | Val Loss: 0.4204
Early stopping triggered after 15 epochs.
Training complete. Loading the best model weights.


In [9]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(model, test_loader, device=None):
    """
    Evaluates the EEGWaveNet model on the test dataset.
    Calculates Accuracy, Binary F1-Score, Weighted F1-Score, Sensitivity, and Specificity.
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
    model.to(device)
    model.eval()
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Ensure correct 4D shape (Batch, Channels, 1, Timepoints)
            if len(inputs.shape) == 3:
                inputs = inputs.unsqueeze(2)
            
            outputs = model(inputs)
            
            # Since the model uses LogSoftmax, argmax gives the predicted class index
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    
    # 1. Accuracy
    accuracy = accuracy_score(all_targets, all_preds)
    
    # 2. F1 Scores (Binary and Weighted as referenced in the paper)
    # Assuming class 1 represents the seizure/positive class
    f1_bin = f1_score(all_targets, all_preds, average='binary', zero_division=0)
    f1_weight = f1_score(all_targets, all_preds, average='weighted', zero_division=0)
    
    # 3. Sensitivity and Specificity via Confusion Matrix
    cm = confusion_matrix(all_targets, all_preds)
    
    # Handle the case where the test set might only contain one class
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    else:
        # Fallbacks if only 1 class is present in the test targets
        sensitivity = 0.0
        specificity = 0.0
        print("Warning: Confusion matrix is not 2x2. Test set may be missing a class.")

    # Print nicely formatted results
    print("--- Test Set Evaluation ---")
    print(f"Accuracy:          {accuracy * 100:.2f}%")
    print(f"F1-Score (Binary): {f1_bin * 100:.2f}%")
    print(f"F1-Score (Weight): {f1_weight * 100:.2f}%")
    print(f"Sensitivity:       {sensitivity * 100:.2f}%")
    print(f"Specificity:       {specificity * 100:.2f}%")
    print("---------------------------")
    
    return {
        'accuracy': accuracy,
        'f1_binary': f1_bin,
        'f1_weighted': f1_weight,
        'sensitivity': sensitivity,
        'specificity': specificity
    }

# --- Example Usage ---
metrics = evaluate_model(model, test_loader)

--- Test Set Evaluation ---
Accuracy:          85.27%
F1-Score (Binary): 85.62%
F1-Score (Weight): 85.27%
Sensitivity:       87.67%
Specificity:       82.88%
---------------------------


In [10]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = get_split(X, y, patient, test_patients=[0])
train_loader, val_loader, test_loader = create_dataloaders(
     X_train, y_train, 
     X_val, y_val, 
     X_test, y_test,      
     batch_size=512
)

model = EEGWaveNet(21, timepoints=768)
trained_model = train_model(model=model, train_loader=train_loader, val_loader=val_loader)
metrics = evaluate_model(model, test_loader)

Epoch 001/50 | Train Loss: 0.6708 | Val Loss: 0.6569
Epoch 002/50 | Train Loss: 0.6042 | Val Loss: 0.6059
Epoch 003/50 | Train Loss: 0.5627 | Val Loss: 0.5851
Epoch 004/50 | Train Loss: 0.5278 | Val Loss: 0.5734
Epoch 005/50 | Train Loss: 0.4616 | Val Loss: 0.5993
Epoch 006/50 | Train Loss: 0.3593 | Val Loss: 0.6141
Epoch 007/50 | Train Loss: 0.2721 | Val Loss: 0.5678
Epoch 008/50 | Train Loss: 0.2036 | Val Loss: 0.4549
Epoch 009/50 | Train Loss: 0.1559 | Val Loss: 0.3729
Epoch 010/50 | Train Loss: 0.1206 | Val Loss: 0.4025
Epoch 011/50 | Train Loss: 0.1073 | Val Loss: 0.4005
Epoch 012/50 | Train Loss: 0.0882 | Val Loss: 0.3976
Epoch 013/50 | Train Loss: 0.0781 | Val Loss: 0.4542
Epoch 014/50 | Train Loss: 0.0699 | Val Loss: 0.4318
Early stopping triggered after 14 epochs.
Training complete. Loading the best model weights.
--- Test Set Evaluation ---
Accuracy:          88.36%
F1-Score (Binary): 88.03%
F1-Score (Weight): 88.35%
Sensitivity:       85.62%
Specificity:       91.10%
-------

In [11]:
import numpy as np
import torch

def run_n_experiments(X, y, patient, test_patients=[0], n_runs=5, batch_size=512, timepoints=768, epochs=50):
    """
    Runs the data splitting, dataloader creation, model training, and evaluation pipeline N times.
    Calculates and reports mean and standard deviation for all evaluation metrics.
    """
    # Dictionary to aggregate metrics across all N runs
    all_metrics = {
        'accuracy': [],
        'f1_binary': [],
        'f1_weighted': [],
        'sensitivity': [],
        'specificity': []
    }

    for run in range(n_runs):
        print(f"\n==================================================")
        print(f"                 RUN {run + 1} / {n_runs}                 ")
        print(f"==================================================")
        
        # Set distinct seeds for each run to vary train/val split and model weight initialization
        seed = 42 + run
        torch.manual_seed(seed)
        np.random.seed(seed)

        # 1. Split Data
        (X_train, y_train), (X_val, y_val), (X_test, y_test) = get_split(
            X, y, patient, test_patients=test_patients, random_state=seed
        )

        # 2. Create DataLoaders
        train_loader, val_loader, test_loader = create_dataloaders(
            X_train, y_train, 
            X_val, y_val, 
            X_test, y_test,      
            batch_size=batch_size
        )

        # 3. Instantiate Fresh Model
        model = EEGWaveNet(in_channels=21, timepoints=timepoints)

        # 4. Train Model
        trained_model = train_model(
            model=model, 
            train_loader=train_loader, 
            val_loader=val_loader,
            epochs=epochs
        )

        # 5. Evaluate Model on Test Set
        run_metrics = evaluate_model(trained_model, test_loader)

        # 6. Record Results
        for metric_name in all_metrics:
            all_metrics[metric_name].append(run_metrics[metric_name])

    # Summary Report
    print("\n" + "=" * 50)
    print(f"    FINAL AGGREGATED METRICS ACROSS {n_runs} RUNS    ")
    print("=" * 50)
    
    summary_metrics = {}
    for metric_name, values in all_metrics.items():
        mean_val = np.mean(values) * 100
        std_val = np.std(values) * 100
        summary_metrics[f"{metric_name}_mean"] = mean_val
        summary_metrics[f"{metric_name}_std"] = std_val
        
        formatted_name = metric_name.replace('_', ' ').title()
        print(f"{formatted_name:<20}: {mean_val:.2f}% ± {std_val:.2f}%")
        
    print("=" * 50)

    return summary_metrics, all_metrics

# --- Example Usage ---
summary, raw_history = run_n_experiments(
     X, y, patient, 
     test_patients=[5], 
     n_runs=10, 
     batch_size=512, 
     timepoints=768
 )


                 RUN 1 / 10                 
Epoch 001/50 | Train Loss: 0.6705 | Val Loss: 0.6288
Epoch 002/50 | Train Loss: 0.5858 | Val Loss: 0.5627
Epoch 003/50 | Train Loss: 0.5548 | Val Loss: 0.5491
Epoch 004/50 | Train Loss: 0.5251 | Val Loss: 0.5637
Epoch 005/50 | Train Loss: 0.4716 | Val Loss: 0.5549
Epoch 006/50 | Train Loss: 0.3891 | Val Loss: 0.5734
Epoch 007/50 | Train Loss: 0.3035 | Val Loss: 0.5563
Epoch 008/50 | Train Loss: 0.2489 | Val Loss: 0.4242
Epoch 009/50 | Train Loss: 0.1949 | Val Loss: 0.3986
Epoch 010/50 | Train Loss: 0.1726 | Val Loss: 0.4052
Epoch 011/50 | Train Loss: 0.1433 | Val Loss: 0.3988
Epoch 012/50 | Train Loss: 0.1383 | Val Loss: 0.4215
Epoch 013/50 | Train Loss: 0.1250 | Val Loss: 0.4179
Epoch 014/50 | Train Loss: 0.1624 | Val Loss: 0.4526
Early stopping triggered after 14 epochs.
Training complete. Loading the best model weights.
--- Test Set Evaluation ---
Accuracy:          52.08%
F1-Score (Binary): 43.90%
F1-Score (Weight): 51.04%
Sensitivity: 

# Patient 0

In [14]:
# --- Example Usage ---
summary, raw_history = run_n_experiments(
     X, y, patient, 
     test_patients=[0], 
     n_runs=10, 
     batch_size=512, 
     timepoints=768
 )


                 RUN 1 / 10                 
Epoch 001/50 | Train Loss: 0.6726 | Val Loss: 0.6526
Epoch 002/50 | Train Loss: 0.5928 | Val Loss: 0.6038
Epoch 003/50 | Train Loss: 0.5583 | Val Loss: 0.5985
Epoch 004/50 | Train Loss: 0.5331 | Val Loss: 0.6244
Epoch 005/50 | Train Loss: 0.4796 | Val Loss: 0.6139
Epoch 006/50 | Train Loss: 0.3840 | Val Loss: 0.6224
Epoch 007/50 | Train Loss: 0.3009 | Val Loss: 0.6539
Epoch 008/50 | Train Loss: 0.2401 | Val Loss: 0.4787
Epoch 009/50 | Train Loss: 0.1838 | Val Loss: 0.4071
Epoch 010/50 | Train Loss: 0.1433 | Val Loss: 0.4031
Epoch 011/50 | Train Loss: 0.1234 | Val Loss: 0.4090
Epoch 012/50 | Train Loss: 0.1015 | Val Loss: 0.4203
Epoch 013/50 | Train Loss: 0.1075 | Val Loss: 0.5232
Epoch 014/50 | Train Loss: 0.0851 | Val Loss: 0.4541
Epoch 015/50 | Train Loss: 0.0712 | Val Loss: 0.4545
Early stopping triggered after 15 epochs.
Training complete. Loading the best model weights.
--- Test Set Evaluation ---
Accuracy:          87.67%
F1-Score (Bi